2.4. Tính bằng Python

In [2]:
def bayes(tien_nghiem, kha_nang, kha_nang_nguoc):

  tu_so = kha_nang * tien_nghiem
  mau_so = tu_so + kha_nang_nguoc * (1 - tien_nghiem)
  return tu_so / mau_so

print("Vi du 1: xet nghiem benh")
print(" Ty le mac benh 1 phan tram, xet nghiem dung 99 phan tram.")
p = bayes(tien_nghiem=0.01, kha_nang=0.99, kha_nang_nguoc=0.01)
print(f" Duong tinh thi kha nang co benh that = {p:.4f}")
print()

print("Cung xet nghiem do, doi ty le mac benh:")
print(" Ty le mac benh Hau nghiem khi duong tinh")
for ty_le in [0.001, 0.01, 0.05, 0.10, 0.50]:
  print(f" {ty_le:6.3f} {bayes(ty_le, 0.99, 0.01):.4f}")
print()

print("Vi du 2: hai day chuyen san xuat")
print(" Day A lam 80 phan tram san pham, ty le loi 1 phan tram.")
print(" Day B lam 20 phan tram san pham, ty le loi 5 phan tram.")
p_b = bayes(tien_nghiem=0.20, kha_nang=0.05, kha_nang_nguoc=0.01)
print(f" Nhat duoc san pham loi, kha nang no tu day B = {p_b:.4f}")
print()

print("Kiem lai bang cach dem tren 1000 san pham:")
loi_a = 800 * 0.01
loi_b = 200 * 0.05
tong_loi = loi_a + loi_b
print(f" Day A: 800 san pham, {loi_a:.0f} cai loi")
print(f" Day B: 200 san pham, {loi_b:.0f} cai loi")
print(f" Ty le tu day B = {loi_b} / {tong_loi} = {loi_b / tong_loi:.4f}")

Vi du 1: xet nghiem benh
 Ty le mac benh 1 phan tram, xet nghiem dung 99 phan tram.
 Duong tinh thi kha nang co benh that = 0.5000

Cung xet nghiem do, doi ty le mac benh:
 Ty le mac benh Hau nghiem khi duong tinh
  0.001 0.0902
  0.010 0.5000
  0.050 0.8390
  0.100 0.9167
  0.500 0.9900

Vi du 2: hai day chuyen san xuat
 Day A lam 80 phan tram san pham, ty le loi 1 phan tram.
 Day B lam 20 phan tram san pham, ty le loi 5 phan tram.
 Nhat duoc san pham loi, kha nang no tu day B = 0.5556

Kiem lai bang cach dem tren 1000 san pham:
 Day A: 800 san pham, 8 cai loi
 Day B: 200 san pham, 10 cai loi
 Ty le tu day B = 10.0 / 18.0 = 0.5556


3.3. Làm bằng scikit‐learn

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
df = pd.read_csv("tin_nhan.csv")
print("Kich thuoc bang:", df.shape)
print()
print("So tin theo nhan (1 la rac, 0 la thuong):")
print(df["la_rac"].value_counts())
print()
print("Ty le tin rac:", round(df["la_rac"].mean(), 4))
print()

print("Ba tin nhan dau tien:")
for i in range(3):
  print(f" [{df.la_rac[i]}] {df.noi_dung[i]}")
print()

vec = CountVectorizer(token_pattern=r"(?u)\{}S+")
X = vec.fit_transform(df["noi_dung"])

print("Kich thuoc bang dem tu:", X.shape)
print("So tu trong tu dien:", len(vec.vocabulary_))
print()

dem = pd.DataFrame(X.toarray(), columns=vec.get_feature_names_out())
dem["la_rac"] = df["la_rac"]
tong = dem.groupby("la_rac").sum().T
tong.columns = ["thuong", "rac"]

print("Muoi tu xuat hien nhieu nhat trong tin rac:")
print(tong.sort_values("rac", ascending=False).head(10))
print()
print("Muoi tu xuat hien nhieu nhat trong tin thuong:")
print(tong.sort_values("thuong", ascending=False).head(10))

4.2. Huấn luyện và chấm điểm

In [14]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import (accuracy_score, confusion_matrix,precision_score, recall_score)
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB

df = pd.read_csv("tin_nhan.csv")
vec = CountVectorizer(token_pattern=r"(?u)\S+")
X = vec.fit_transform(df["noi_dung"])
y = df["la_rac"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=17, stratify=y)
print("So tin de hoc :", X_train.shape[0])
print("So tin de kiem tra:", X_test.shape[0])
print()

mo_hinh = MultinomialNB(alpha=1.0)
mo_hinh.fit(X_train, y_train)
y_pred = mo_hinh.predict(X_test)

tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
print("Ma tran nham lan")
print(f" TN = {tn:2d} thu thuong duoc giu dung")
print(f" FP = {fp:2d} thu thuong bi xoa nham")
print(f" FN = {fn:2d} thu rac lot luoi")
print(f" TP = {tp:2d} thu rac bi chan dung")
print()

print(f"Accuracy = {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision = {precision_score(y_test, y_pred):.4f}")
print(f"Recall = {recall_score(y_test, y_pred):.4f}")
print()

tin_moi = [
    "trung thuong khuyen mai nhanh tay click ngay",
    "hop nhom bao cao chieu mai phong 305",
    "ban gui giup em tai lieu bai tap nhe",
]
X_moi = vec.transform(tin_moi)
p_rac = mo_hinh.predict_proba(X_moi)[:, 1]

print("Thu tren ba tin nhan tu go:")
for tin, p in zip(tin_moi, p_rac):
  nhan = "RAC" if p >= 0.5 else "thuong"
  print(f" [{nhan:6s}] p = {p:.4f} <‐ {tin}")

So tin de hoc : 225
So tin de kiem tra: 75

Ma tran nham lan
 TN = 45 thu thuong duoc giu dung
 FP =  0 thu thuong bi xoa nham
 FN =  4 thu rac lot luoi
 TP = 26 thu rac bi chan dung

Accuracy = 0.9467
Precision = 1.0000
Recall = 0.8667

Thu tren ba tin nhan tu go:
 [RAC   ] p = 0.9992 <‐ trung thuong khuyen mai nhanh tay click ngay
 [thuong] p = 0.0983 <‐ hop nhom bao cao chieu mai phong 305
 [thuong] p = 0.0719 <‐ ban gui giup em tai lieu bai tap nhe


5.3. Tự cài đặt để thấy tận mắt

In [18]:
from collections import Counter
kho = [
    ("trung thuong ngay", 1),
    ("khuyen mai giam gia", 1),
    ("trung thuong click", 1),
    ("hop nhom chieu mai", 0),
    ("bai tap nhom nop mai", 0),
    ("thay gui tai lieu", 0),
]
tu_dien = sorted({t for cau, _ in kho for t in cau.split()})
V = len(tu_dien)
print(f"Tu dien co {V} tu:")
for i in range(0, V, 9):
  print(" " + " ".join(tu_dien[i:i + 9]))
print()

dem = {0: Counter(), 1: Counter()}
tong = {0: 0, 1: 0}
for cau, nhan in kho:
  for t in cau.split():
    dem[nhan][t] += 1
    tong[nhan] += 1

print("So lan xuat hien cua tung tu")
print(f" {'tu':12s} {'rac':>5s} {'thuong':>7s}")
for t in tu_dien:
  print(f" {t:12s} {dem[1][t]:5d} {dem[0][t]:7d}")
print()

def xac_suat(tu, nhan, alpha):
  return (dem[nhan][tu] + alpha) / (tong[nhan] + alpha * V)

def diem(cau, nhan, alpha):
  p = 0.5
  for t in cau.split():
    if t in tu_dien:
      p *= xac_suat(t, nhan, alpha)
  return p

cau_thu = "trung thuong bai tap"
print(f'Tin nhan thu: "{cau_thu}"')
print("Tu 'bai' chi xuat hien o lop thuong, chua bao gio o lop rac.")
print()

for alpha in [0.0, 1.0]:
  d_rac = diem(cau_thu, 1, alpha)
  d_thuong = diem(cau_thu, 0, alpha)
  ket = "rac" if d_rac > d_thuong else "thuong"
  print(f"alpha = {alpha}")
  print(f" diem lop rac = {d_rac:.10f}")
  print(f" diem lop thuong = {d_thuong:.10f}")
  print(f" ket luan = {ket}")
  print()
print("Khong lam tron, CA HAI diem deu bang 0 nen mo hinh te liet.")
print("Lop rac bang 0 vi tu 'bai' chua tung xuat hien o lop rac.")
print("Lop thuong bang 0 vi tu 'trung' chua tung xuat hien o lop thuong.")
print("Co lam tron, diem lop rac lon hon va mo hinh ket luan dung.")

Tu dien co 18 tu:
 bai chieu click gia giam gui hop khuyen lieu
 mai ngay nhom nop tai tap thay thuong trung

So lan xuat hien cua tung tu
 tu             rac  thuong
 bai              0       1
 chieu            0       1
 click            1       0
 gia              1       0
 giam             1       0
 gui              0       1
 hop              0       1
 khuyen           1       0
 lieu             0       1
 mai              1       2
 ngay             1       0
 nhom             0       2
 nop              0       1
 tai              0       1
 tap              0       1
 thay             0       1
 thuong           2       0
 trung            2       0

Tin nhan thu: "trung thuong bai tap"
Tu 'bai' chi xuat hien o lop thuong, chua bao gio o lop rac.

alpha = 0.0
 diem lop rac = 0.0000000000
 diem lop thuong = 0.0000000000
 ket luan = thuong

alpha = 1.0
 diem lop rac = 0.0000073212
 diem lop thuong = 0.0000021656
 ket luan = rac

Khong lam tron, CA HAI diem deu bang 0 nen mo 

6.2. So với hồi quy logistic trên cùng dữ liệu

In [21]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB

df = pd.read_csv("sinh_vien (1).csv")
X = df[["gio_on", "diem_giua_ky"]]
y = df["qua_mon"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=17, stratify=y)

for ten, mo_hinh in [("Naive Bayes ", GaussianNB()),
                     ("Hoi quy logistic", LogisticRegression())]:
    mo_hinh.fit(X_train, y_train)
    acc = accuracy_score(y_test, mo_hinh.predict(X_test))
    print(f"{ten}: accuracy = {acc:.4f}")
print()

gnb = GaussianNB().fit(X_train, y_train)
print("Nhung gi Naive Bayes hoc duoc")
print(f" {'':16s} {'gio_on':>10s} {'diem_giua_ky':>14s}")
for i, ten_lop in enumerate(["Lop 0 (rot) ", "Lop 1 (qua) "]):
    tb = gnb.theta_[i]
    dl = gnb.var_[i] ** 0.5
    print(f" {ten_lop} trung binh {tb[0]:10.2f} {tb[1]:14.2f}")
    print(f" {'':12s} do lech {dl[0]:10.2f} {dl[1]:14.2f}")
print()

ty_le = ", ".join(f"{float(v):.4f}" for v in gnb.class_prior_)
print(f"Ty le tung lop trong tap hoc: {ty_le}")
print()

ban_moi = pd.DataFrame({"gio_on": [5, 13, 20, 28],
                          "diem_giua_ky": [4.0, 6.0, 7.0, 9.0]})
p = gnb.predict_proba(ban_moi)[:, 1]
print("Du doan cho bon ban moi:")
print(f" {'gio on':>8s} {'diem GK':>9s} {'P(qua mon)':>12s}")
for (_, r), pi in zip(ban_moi.iterrows(), p):
  print(f" {r.gio_on:8.0f} {r.diem_giua_ky:9.1f} {pi:12.4f}")

Naive Bayes : accuracy = 0.9000
Hoi quy logistic: accuracy = 0.9000

Nhung gi Naive Bayes hoc duoc
                      gio_on   diem_giua_ky
 Lop 0 (rot)  trung binh       8.18           3.99
              do lech       5.29           1.92
 Lop 1 (qua)  trung binh      20.03           6.70
              do lech       5.81           1.50

Ty le tung lop trong tap hoc: 0.4111, 0.5889

Du doan cho bon ban moi:
   gio on   diem GK   P(qua mon)
        5       4.0       0.0137
       13       6.0       0.6550
       20       7.0       0.9855
       28       9.0       0.9999


Bài tập 1: Bayes với ba nguyên nhân

In [3]:
def bayes_nhieu_nhanh(tien_nghiem, kha_nang):
    """Xac suat hau nghiem cho nhieu nguyen nhan.
    tien_nghiem: P(nhom); kha_nang: P(bang chung | nhom)."""
    tich = [pn * kn for pn, kn in zip(tien_nghiem, kha_nang)]
    mau_so = sum(tich)                      # P(bang chung) = P(qua mon)
    return [t / mau_so for t in tich]

ten_nhom = ["hoc deu", "hoc don", "it hoc"]
ty_le_nhom = [0.50, 0.30, 0.20]             # tien nghiem
ty_le_qua = [0.95, 0.70, 0.30]              # P(qua mon | nhom)
hau_nghiem = bayes_nhieu_nhanh(ty_le_nhom, ty_le_qua)
for ten, p in zip(ten_nhom, hau_nghiem):
    print(f"P({ten:7s} | qua mon) = {p:.4f}")
print(f"Tong ba xac suat = {sum(hau_nghiem):.4f}")

P(hoc deu | qua mon) = 0.6376
P(hoc don | qua mon) = 0.2819
P(it hoc  | qua mon) = 0.0805
Tong ba xac suat = 1.0000


Mẫu số chung là P(qua môn) = 0.5×0.95 + 0.3×0.70 + 0.2×0.30 = 0.745. Mỗi nhóm lấy phần đóng góp của mình (tiên nghiệm × khả năng) chia cho tổng đó, nên ba kết quả chắc chắn cộng lại bằng 1. Biết một bạn đã qua môn làm niềm tin dịch về nhóm học đều (0.50 lên 0.6376), còn nhóm ít học tụt từ 0.20 xuống 0.0805

Bài tập 2: Từ nào tố cáo mạnh nhất

In [5]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

df = pd.read_csv("tin_nhan.csv")
vec = CountVectorizer(token_pattern=r"(?u)\S+")
X = vec.fit_transform(df["noi_dung"]); y = df["la_rac"]
mo_hinh = MultinomialNB(alpha=1.0).fit(X, y)

tu = np.array(vec.get_feature_names_out())
chenh_lech = mo_hinh.feature_log_prob_[1] - mo_hinh.feature_log_prob_[0]
thu_tu = np.argsort(chenh_lech)

print("5 tu to cao rac manh nhat:")
for i in thu_tu[::-1][:5]:
    print(f"  {tu[i]:10s} {chenh_lech[i]:+.4f} (ti so {np.exp(chenh_lech[i]):.1f} lan)")
print("5 tu cho thay tin thuong manh nhat:")
for i in thu_tu[:5]:
    print(f"  {tu[i]:10s} {chenh_lech[i]:+.4f} (ti so {np.exp(chenh_lech[i]):.2f} lan)")

5 tu to cao rac manh nhat:
  mãi        +3.2034 (ti so 24.6 lan)
  tay        +2.9056 (ti so 18.3 lan)
  ngay       +2.6618 (ti so 14.3 lan)
  miễn       +2.4795 (ti so 11.9 lan)
  trúng      +2.3742 (ti so 10.7 lan)
5 tu cho thay tin thuong manh nhat:
  lớp        -2.6122 (ti so 0.07 lan)
  em         -2.5618 (ti so 0.08 lan)
  bài        -2.2946 (ti so 0.10 lan)
  báo        -2.2741 (ti so 0.10 lan)
  phòng      -2.2741 (ti so 0.10 lan)


Hiệu hai log xác suất chính là log của tỉ số P(từ | rác) / P(từ | thường). Từ “mãi” xuất hiện ở tin rác nhiều hơn tin thường khoảng 24.6 lần. Các từ rác (mãi, tay, ngay, miễn, trúng) mang giọng quảng cáo, còn các từ thường (lớp, em, bài, báo, phòng) thuộc đời sống học đường

Bài tập 3: Đổi tham số làm trơn

In [6]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB

df = pd.read_csv("tin_nhan.csv")
X = CountVectorizer(token_pattern=r"(?u)\S+").fit_transform(df["noi_dung"])
y = df["la_rac"]
X_hoc, X_kt, y_hoc, y_kt = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y)

print("  alpha   accuracy  precision  recall")
for alpha in [0.001, 0.1, 1.0, 10.0, 100.0]:
    y_du_doan = MultinomialNB(alpha=alpha).fit(X_hoc, y_hoc).predict(X_kt)
    print(f"{alpha:7g}   {accuracy_score(y_kt, y_du_doan):.4f}    "
          f"{precision_score(y_kt, y_du_doan, zero_division=0):.4f}    "
          f"{recall_score(y_kt, y_du_doan):.4f}")

  alpha   accuracy  precision  recall
  0.001   0.9467    1.0000    0.8667
    0.1   0.9467    1.0000    0.8667
      1   0.9467    1.0000    0.8667
     10   0.9467    1.0000    0.8667
    100   0.9467    1.0000    0.8667


Trong khoảng alpha từ 0.001 đến 100, kết quả không đổi. Từ điển chỉ có 40 từ và mỗi lớp có hơn 700 lần từ xuất hiện trong tập học (1064 lớp thường, 710 lớp rác), nên làm trơn nhẹ hay vừa không đổi thứ tự các từ

Bài tập 4: Tự viết bộ lọc không dùng thư viện

In [7]:
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB

df = pd.read_csv("tin_nhan.csv")
tin_hoc, tin_kt, nhan_hoc, nhan_kt = train_test_split(
    df["noi_dung"], df["la_rac"], test_size=0.25, random_state=17,
    stratify=df["la_rac"])

ALPHA = 1.0
# Tu dien lay tu toan bo 300 tin (giong CountVectorizer o muc 4)
tu_dien = sorted({tu for tin in df["noi_dung"] for tu in tin.split()})
so_tu_dien = len(tu_dien)

# Buoc 1: tu dem so lan xuat hien cua tung tu o moi lop, CHI tren tap hoc
dem_tu = {0: Counter(), 1: Counter()}
tong_tu = {0: 0, 1: 0}
so_tin = {0: 0, 1: 0}
for tin, nhan in zip(tin_hoc, nhan_hoc):
    so_tin[nhan] += 1
    for tu in tin.split():
        dem_tu[nhan][tu] += 1
        tong_tu[nhan] += 1

# Buoc 2: tien nghiem va xac suat tu co lam tron Laplace, luu dang LOGA
log_tien_nghiem = {c: np.log(so_tin[c] / len(tin_hoc)) for c in (0, 1)}
log_xac_suat_tu = {
    c: {tu: np.log((dem_tu[c][tu] + ALPHA) / (tong_tu[c] + ALPHA * so_tu_dien))
        for tu in tu_dien}
    for c in (0, 1)}

# Buoc 3: diem = log tien nghiem + TONG log xac suat tu (khong nhan tich)
def diem_lop(tin, lop):
    return log_tien_nghiem[lop] + sum(
        log_xac_suat_tu[lop][tu] for tu in tin.split() if tu in log_xac_suat_tu[lop])

def du_doan(tin):
    return int(diem_lop(tin, 1) > diem_lop(tin, 0))

nhan_tu_viet = [du_doan(tin) for tin in tin_kt]
acc_tu_viet = accuracy_score(nhan_kt, nhan_tu_viet)

# Doi chieu voi thu vien
X = CountVectorizer(token_pattern=r"(?u)\S+").fit_transform(df["noi_dung"])
X_hoc, X_kt, y_hoc, y_kt = train_test_split(
    X, df["la_rac"], test_size=0.25, random_state=17, stratify=df["la_rac"])
nhan_thu_vien = MultinomialNB(alpha=ALPHA).fit(X_hoc, y_hoc).predict(X_kt)

print(f"Accuracy tu viet : {acc_tu_viet:.4f}")
print(f"Accuracy thu vien: {accuracy_score(y_kt, nhan_thu_vien):.4f}")
print("So tin du doan khac nhau:", int((np.array(nhan_tu_viet) != nhan_thu_vien).sum()))

Accuracy tu viet : 0.9467
Accuracy thu vien: 0.9467
So tin du doan khac nhau: 0


Bài tập 5: Chọn sai biến thể Naive Bayes

In [10]:
import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB, MultinomialNB

df = pd.read_csv("sinh_vien (1).csv")
X = df[["gio_on", "diem_giua_ky"]]; y = df["qua_mon"]
X_hoc, X_kt, y_hoc, y_kt = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y)

for ten, mo_hinh in [("MultinomialNB", MultinomialNB()), ("GaussianNB   ", GaussianNB())]:
    mo_hinh.fit(X_hoc, y_hoc)
    print(f"{ten}: accuracy = {accuracy_score(y_kt, mo_hinh.predict(X_kt)):.4f}")

MultinomialNB: accuracy = 0.8000
GaussianNB   : accuracy = 0.9000


Chương trình không báo lỗi, vì hai cột chỉ chứa số không âm, đúng điều kiện duy nhất mà MultinomialNB kiểm tra. Accuracy ra 0.8000, thấp hơn 0.9000 của GaussianNB ở Mục 6

Bài tập 6: Lớp hiếm và xác suất tiên nghiệm

In [11]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score, confusion_matrix, recall_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB

df = pd.read_csv("tin_nhan.csv")
tin_thuong = df[df["la_rac"] == 0]                  # 180 tin thuong
tin_rac_dau = df[df["la_rac"] == 1].head(20)        # chi 20 tin rac dau tien
bo_mat_can_bang = pd.concat([tin_thuong, tin_rac_dau], ignore_index=True)

X = CountVectorizer(token_pattern=r"(?u)\S+").fit_transform(bo_mat_can_bang["noi_dung"])
y = bo_mat_can_bang["la_rac"]
X_hoc, X_kt, y_hoc, y_kt = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y)

mo_hinh = MultinomialNB().fit(X_hoc, y_hoc)
y_du_doan = mo_hinh.predict(X_kt)
tn, fp, fn, tp = confusion_matrix(y_kt, y_du_doan, labels=[0, 1]).ravel()

print(f"Accuracy       = {accuracy_score(y_kt, y_du_doan):.4f}")
print(f"Recall lop rac = {recall_score(y_kt, y_du_doan, pos_label=1):.4f}")
print(f"TN={tn} FP={fp} FN={fn} TP={tp}")
print(f"Doan bua 'tin nao cung thuong': accuracy = {1 - y_kt.mean():.4f}, recall = 0")
print("Tien nghiem (thuong, rac):", np.exp(mo_hinh.class_log_prior_).round(4))

Accuracy       = 0.9800
Recall lop rac = 0.8000
TN=45 FP=0 FN=1 TP=4
Doan bua 'tin nao cung thuong': accuracy = 0.9000, recall = 0
Tien nghiem (thuong, rac): [0.9 0.1]


Accuracy không còn phản ánh đúng chất lượng bộ lọc. 0.9800 nghe rất cao, thậm chí cao hơn 0.9467 của bộ cân bằng, nhưng điều đó không có nghĩa là bộ lọc tốt hơn. Một bộ lọc ngu ngốc chặn không tin nào đã đạt 0.9000 mà không bắt được tin rác nào (recall bằng 0). Accuracy bị 45 tin thường dễ đoán nâng lên, trong khi việc quan trọng là bắt tin rác chỉ lộ ra ở recall: bộ lọc để lọt 1 trong 5 tin rác, tức recall 0.8000